# Production-Grade Zero-Cost SOTA RAG Pipeline

**Goal:** Implement a State-of-the-Art RAG pipeline for trading books using 100% free tools.

### The Stack
1.  **LLM:** Groq (Llama 3.3 70B) - *Free Tier (14.4k req/day)*
2.  **Cross-Book Reasoning:** LightRAG + Ollama (Local) - *Open Source*
3.  **Embeddings:** BGE-Large - *HuggingFace (Free)*
4.  **Hybrid Retrieval:** BM25 (Keyword) + Chroma (Dense) + RRF Fusion
5.  **Reranking:** BGE-Reranker-Large - *HuggingFace (Free)*
6.  **Evaluation:** RAGAS (configured with Groq) - *Open Source*

### Production Hardening Features:
* **Concurrency Safety:** File locking prevents race conditions on manifest updates (graceful fallback on Windows).
* **Data Integrity:** Atomic saves, sorted checkpoints, and robust JSON loading.
* **Garbled Text Detection:** Smart Unicode filter for OCR errors.
* **Memory Safety:** Explicit GC, batched processing, checkpoint size guards, and VRAM cleanup.
* **Fault Tolerance:** Robust handling of PDF encryption (with timeouts), API timeouts, and missing data.

## 1. Environment Setup & Validation

In [ ]:
# Install necessary libraries
!pip install -q groq==0.11.0 pypdf==5.0.1 chromadb==0.5.5 sentence-transformers==3.1.1 ragas==0.1.21 lightrag-hku==1.0.6 langchain-groq==0.2.0 langchain-community==0.2.16 nest_asyncio==1.6.0 tenacity==9.0.0 tqdm==4.66.5 datasets==2.21.0 torch==2.4.1 rank-bm25==0.2.2

In [ ]:
import os
import time
import json
import asyncio
import logging
import nest_asyncio
import requests
import subprocess
import sys
import getpass
import torch
import gc
from typing import List, Dict, Optional, Any, Tuple
from pathlib import Path
from tqdm.auto import tqdm
from tenacity import retry, stop_after_attempt, wait_random_exponential, retry_if_exception_type, before_sleep_log

# Apply nest_asyncio to allow nested event loops (critical for LightRAG in Jupyter)
nest_asyncio.apply()

# Ensure local package import path
PROJECT_ROOT = Path(".").resolve()
SRC_DIR = PROJECT_ROOT / "src"
if SRC_DIR.exists() and str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from rag_pipeline.config import load_settings
from rag_pipeline.manifest import (
    load_manifest as load_manifest_pkg,
    remove_manifest_entries as remove_manifest_entries_pkg,
    update_manifest as update_manifest_pkg,
    update_manifest_bulk as update_manifest_bulk_pkg,
)
from rag_pipeline.observability import configure_structured_logging, log_event
from rag_pipeline.retrieval import top_k_indices_desc
from rag_pipeline.storage import save_json_atomic as save_json_atomic_pkg

# --- CONFIGURATION & CONSTANTS ---
# Paths (Dynamic for Colab Persistence)
if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = Path("/content/drive/MyDrive/TradeRAG")
else:
    BASE_DIR = Path(".")

settings = load_settings(base_dir=BASE_DIR)

# Pipeline Control
PIPELINE_VERSION = settings.pipeline_version

# Paths
DATA_DIR = settings.data_dir
PROCESSED_DIR = settings.processed_dir
DB_DIR = settings.db_dir
LIGHTRAG_DIR = settings.lightrag_dir

# Pipeline Tuning Constants
GROQ_TIMEOUT = settings.groq_timeout
RETRIEVAL_TOP_K = settings.retrieval_top_k
RRF_K = settings.rrf_k
RERANK_CANDIDATES = settings.rerank_candidates
FINAL_TOP_K = settings.final_top_k
RERANK_BATCH_SIZE = settings.rerank_batch_size
CONTEXT_MAX_CHARS = settings.context_max_chars
EVAL_THROTTLE_SEC = settings.eval_throttle_sec

# Processing Constants
MIN_CHUNK_CHARS = settings.min_chunk_chars
GC_INTERVAL_PDFS = settings.gc_interval_pdfs
MAX_CHECKPOINT_BYTES = settings.max_checkpoint_bytes

# Secure API Key Input (CI/headless-safe)
if "GROQ_API_KEY" not in os.environ:
    ci_mode = os.environ.get("CI", "").strip().lower() in {"1", "true", "yes"}
    interactive_tty = hasattr(sys.stdin, "isatty") and sys.stdin.isatty()
    if ci_mode or not interactive_tty:
        raise ValueError("GROQ_API_KEY is required in non-interactive mode. Export it before running.")
    print("Please enter your Groq API Key (input will be hidden):")
    key_input = getpass.getpass("Groq API Key:")
    if not key_input.startswith("gsk_"):
        print("WARNING: Key does not start with 'gsk_'. It might be invalid.")
    os.environ["GROQ_API_KEY"] = key_input

# Setup Logging
configure_structured_logging(os.environ.get("LOG_LEVEL", "INFO"))
logger = logging.getLogger("rag_pipeline.notebook")
log_event(logger, "notebook_bootstrap", package_backed=True, base_dir=str(BASE_DIR))

# --- HELPER: CLIENT CACHING ---
_groq_client_instance = None

def get_cached_groq_client():
    """Returns a singleton Groq client to avoid overhead of re-init per call."""
    global _groq_client_instance
    if _groq_client_instance is None:
        api_key = os.environ.get("GROQ_API_KEY")
        if not api_key:
            raise ValueError("GROQ_API_KEY not found in environment. Please run setup cell.")

        import groq
        _groq_client_instance = groq.Groq(api_key=api_key, timeout=GROQ_TIMEOUT)
    return _groq_client_instance

# --- HELPERS: PACKAGE-BACKED WRAPPERS ---
def save_json_atomic(data: Any, filepath: Path):
    save_json_atomic_pkg(data, filepath, logger=logger)


def load_manifest(store_name: str) -> Dict[str, str]:
    return load_manifest_pkg(PROCESSED_DIR, store_name)


def update_manifest(store_name: str, pdf_name: str, file_hash: str):
    update_manifest_pkg(PROCESSED_DIR, store_name, pdf_name, file_hash)


def update_manifest_bulk(store_name: str, updates: Dict[str, str]):
    update_manifest_bulk_pkg(PROCESSED_DIR, store_name, updates)


def remove_manifest_entries(store_name: str, pdf_names):
    remove_manifest_entries_pkg(PROCESSED_DIR, store_name, pdf_names)

# --- SPECIAL GOOGLE COLAB SETUP ---
def setup_colab_ollama():
    """Installs and starts Ollama if running in Google Colab."""
    if "google.colab" in sys.modules:
        print("Detected Google Colab Environment. Installing Ollama...")
        if os.environ.get("ALLOW_REMOTE_INSTALL", "0") != "1":
            print("Skipping automatic Ollama install for security. Set ALLOW_REMOTE_INSTALL=1 to enable.")
            return

        install_script_path = "ollama_install.sh"
        try:
            print("Downloading Ollama install script...")
            resp = requests.get("https://ollama.com/install.sh", timeout=30)
            if resp.status_code == 200:
                with open(install_script_path, "wb") as f:
                    f.write(resp.content)
                print("Executing install script...")
                subprocess.run(["sh", install_script_path], check=True, timeout=600)
            else:
                print(f"Failed to download installer. Status: {resp.status_code}")
                return
        except (requests.exceptions.RequestException, subprocess.SubprocessError, OSError) as e:
            print(f"Setup failed: {e}")
            return

        print("Starting Ollama Server...")
        subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

        for _ in range(20):
            try:
                if requests.get("http://localhost:11434", timeout=2).status_code == 200:
                    print("Ollama is ready!")
                    break
            except (requests.exceptions.ConnectionError, requests.exceptions.Timeout):
                time.sleep(1)
        else:
            print("Warning: Ollama took too long to start.")

        print("Pulling required models...")
        subprocess.run("ollama pull llama3.1".split(), check=True, timeout=1800)
        subprocess.run("ollama pull nomic-embed-text".split(), check=True, timeout=1800)
        print("Ollama setup complete.")

if os.environ.get("AUTO_SETUP_OLLAMA", "0") == "1":
    setup_colab_ollama()


def validate_environment():
    if not os.environ.get("GROQ_API_KEY"):
        raise ValueError("GROQ_API_KEY is missing!")

    pdfs = list(DATA_DIR.glob("*.pdf"))
    if not pdfs:
        logger.warning(f"No PDFs found in {DATA_DIR}. Please add .pdf files.")
    else:
        logger.info(f"Found {len(pdfs)} PDFs in {DATA_DIR}")

    try:
        resp = requests.get("http://localhost:11434", timeout=5)
        if resp.status_code != 200:
             logger.warning("Ollama server responding but not 200 OK.")
    except requests.exceptions.ConnectionError:
        logger.warning("Ollama connection failed. Ensure 'ollama serve' is running.")
    except requests.exceptions.Timeout:
        logger.warning("Ollama connection timed out.")

if os.environ.get("AUTO_VALIDATE_ENV", "0") == "1":
    validate_environment()



## 2. Phase 1: Robust Page-Level Chunking (With Garbage Filtering)
Uses `Chunk.id` (PDF Name + Page) to handle resume logic and checks for garbage text.

In [ ]:
from rag_pipeline.models import Chunk as PipelineChunk
from rag_pipeline.processing import process_pdf_page_level as process_pdf_page_level_pkg, sync_master_chunks
from rag_pipeline.storage import load_chunks_map as load_chunks_map_pkg

Chunk = PipelineChunk


def load_chunks_map(filename: str) -> Dict[str, Chunk]:
    return load_chunks_map_pkg(
        processed_dir=PROCESSED_DIR,
        filename=filename,
        max_checkpoint_bytes=MAX_CHECKPOINT_BYTES,
        chunk_type=Chunk,
        logger=logger,
    )


def process_pdf_page_level(pdf_path: Path, processed_state: Dict[str, Tuple[str, str]]) -> List[Chunk]:
    return process_pdf_page_level_pkg(
        pdf_path,
        processed_state,
        pipeline_version=PIPELINE_VERSION,
        min_chunk_chars=MIN_CHUNK_CHARS,
        garble_threshold=0.40,
        timeout_ctx=None,
        logger=logger,
    )


# --- Main Processing Loop ---
all_pdfs = list(DATA_DIR.glob("*.pdf"))
master_chunks_file = "master_chunks.json"

chunk_map = load_chunks_map(master_chunks_file)
processed_state = {}
pdf_states = {}
for c in chunk_map.values():
    pdf_states.setdefault(c.pdf_name, set()).add((c.file_hash, c.pipeline_version))

for pdf_name, states in pdf_states.items():
    if len(states) == 1:
        processed_state[pdf_name] = next(iter(states))
    else:
        logger.warning(f"Inconsistent checkpoint state for {pdf_name}; forcing reprocess.")

logger.info(f"Resuming with {len(chunk_map)} existing chunks.")

all_chunks = sync_master_chunks(
    tqdm(all_pdfs, desc="Scanning PDFs"),
    chunk_map=chunk_map,
    processed_state=processed_state,
    process_fn=process_pdf_page_level,
    gc_interval_pdfs=GC_INTERVAL_PDFS,
    processed_dir=PROCESSED_DIR,
    master_chunks_file=master_chunks_file,
    logger=logger,
)



## 3. Phase 2: Resilient Context Enrichment
Uses Jittered Exponential Backoff to handle rate limits and invalidates stale enrichment.

In [ ]:
import groq
from rag_pipeline.enrichment import batch_enrich_chunks as batch_enrich_chunks_pkg


@retry(
    wait=wait_random_exponential(min=2, max=60),
    stop=stop_after_attempt(5),
    retry=retry_if_exception_type((
        groq.RateLimitError,
        groq.APIError,
        groq.APIConnectionError,
        groq.InternalServerError,
        requests.exceptions.RequestException,
        requests.exceptions.Timeout
    )),
    before_sleep=before_sleep_log(logger, logging.WARNING)
)
def generate_context_prefix_safe(chunk: Chunk) -> str:
    chunk_text = (chunk.text or "")
    prompt = f"""Generate a 2-3 sentence context for this chunk:
Book: {chunk.pdf_name}
Page: {chunk.page_number}
Chunk:
{chunk_text[:1000]}
Context should explain:
1. What book/chapter this is from
2. The main topic being discussed
3. Key trading concepts mentioned
Return ONLY the context, nothing else."""

    client = get_cached_groq_client()
    chat_completion = client.chat.completions.create(
        messages=[{"role": "user", "content": prompt}],
        model="llama-3.3-70b-versatile",
        temperature=0.3,
        max_tokens=150
    )
    if not chat_completion.choices or not chat_completion.choices[0].message:
        raise ValueError("Invalid response from Groq")
    return chat_completion.choices[0].message.content


def batch_enrich_chunks(chunks: List[Chunk]):
    enriched_file = "enriched_chunks.json"
    enriched_map = load_chunks_map(enriched_file)
    print(f"Enrichment Progress: {len(enriched_map)}/{len(chunks)} chunks in cache.")

    def persist_fn(current_map):
        sorted_data = sorted([c.to_dict() for c in current_map.values()], key=lambda x: x['id'])
        save_json_atomic(sorted_data, PROCESSED_DIR / enriched_file)

    print("Processing enrichment queue...")
    enriched = batch_enrich_chunks_pkg(
        chunks,
        enriched_map=enriched_map,
        generate_context_fn=generate_context_prefix_safe,
        persist_fn=persist_fn,
        logger=logger,
        sleep_seconds=0.2,
        periodic_save_every=20,
    )
    gc.collect()
    return enriched


# enriched_chunks = batch_enrich_chunks(all_chunks)



## 3.5. Phase 2.5: BM25 Index Construction (Hybrid Search Setup)
Builds an in-memory inverted index for keyword search. This must be run after enrichment.

In [ ]:
from rag_pipeline.bm25_utils import build_bm25_index, tokenize_for_bm25


if 'enriched_chunks' in locals() and enriched_chunks:
    print("Building BM25 index...")
    bm25_index, bm25_id_map = build_bm25_index(enriched_chunks, logger=logger)
else:
    _disk_chunks = load_chunks_map("enriched_chunks.json")
    if _disk_chunks:
        print("Loaded chunks from disk for BM25.")
        bm25_index, bm25_id_map = build_bm25_index(list(_disk_chunks.values()), logger=logger)
    else:
        print("No chunks found. BM25 will be empty.")
        bm25_index, bm25_id_map = None, {}



## 4. Phase 3 & 4: Ingestion (Dual Manifests + Cleanup + Commit-on-Success)

In [ ]:
import chromadb
from chromadb.utils import embedding_functions
from lightrag import LightRAG
from lightrag.llm import ollama_model_complete, ollama_embedding
from lightrag.utils import EmbeddingFunc


def populate_chromadb(chunks: List[Chunk]):
    print("Initializing ChromaDB...")
    chroma_client = chromadb.PersistentClient(path=str(DB_DIR))
    emb_fn = embedding_functions.SentenceTransformerEmbeddingFunction(model_name="BAAI/bge-large-en-v1.5")
    collection = chroma_client.get_or_create_collection(name="trading_books", embedding_function=emb_fn)

    manifest = load_manifest("chroma")

    from rag_pipeline.chroma_pipeline import populate_chromadb as populate_chromadb_pkg

    return populate_chromadb_pkg(
        chunks=chunks,
        collection=collection,
        manifest=manifest,
        update_manifest_bulk_fn=lambda store_name, updates: update_manifest_bulk(store_name, updates),
        remove_manifest_entries_fn=lambda store_name, pdf_names: remove_manifest_entries(store_name, pdf_names),
        logger=logger,
        batch_size=50,
    )


async def populate_lightrag(chunks: List[Chunk]):
    # Health Check (Corrected Exception Handling)
    try:
        resp = requests.get("http://localhost:11434/api/tags", timeout=5)
        if resp.status_code != 200:
            raise ConnectionError("Ollama not ready")
            
        # Verify required models exist (safe json access)
        installed_models = [m.get('name', '') for m in resp.json().get('models', []) if isinstance(m, dict)]
        required = ["llama3.1", "nomic-embed-text"]
        # Loose matching for version tags (e.g. llama3.1:latest)
        missing = [req for req in required if not any(req in m for m in installed_models)]
        
        if missing:
             raise ValueError(f"Missing required Ollama models: {missing}")

    except (requests.exceptions.ConnectionError, requests.exceptions.Timeout, ValueError) as e:
        logger.error(f"Cannot start LightRAG: {e}")
        return None

    # Optional full reset for stale graph cleanup
    if os.environ.get("RESET_LIGHTRAG_INDEX", "0") == "1":
        logger.warning("RESET_LIGHTRAG_INDEX=1 -> deleting existing LightRAG working directory before ingestion.")
        try:
            import shutil
            if LIGHTRAG_DIR.exists():
                shutil.rmtree(LIGHTRAG_DIR)
            LIGHTRAG_DIR.mkdir(parents=True, exist_ok=True)
        except OSError as e:
            logger.error(f"Failed to reset LightRAG index directory: {e}")
            return None

    print("Initializing LightRAG...")
    
    async def llm_func(prompt, **kwargs): 
        return await ollama_model_complete(prompt, model_name="llama3.1", **kwargs)
    async def embed_func(texts): 
        return await ollama_embedding(texts, embed_model="nomic-embed-text")

    rag = LightRAG(
        working_dir=str(LIGHTRAG_DIR),
        llm_model_func=llm_func,
        embedding_func=EmbeddingFunc(embedding_dim=768, max_token_size=8192, func=embed_func)
    )

    # Load LightRAG-specific manifest
    manifest = load_manifest("lightrag")
    
    # Group by PDF
    pdf_groups = {}
    for c in chunks:
        pdf_groups.setdefault(c.pdf_name, []).append(c)
    
    pdfs_to_process = {}
    for pdf, grp in pdf_groups.items():
        file_hash = grp[0].file_hash
        if pdf not in manifest or manifest[pdf] != file_hash:
            pdfs_to_process[pdf] = grp

    print(f"Ingesting {len(pdfs_to_process)} documents into Knowledge Graph...")
    
    for i, (pdf, grp) in enumerate(tqdm(pdfs_to_process.items(), desc="LightRAG Ingestion")):
        # NARRATIVE SORTING: Ensure pages are processed in order
        grp.sort(key=lambda x: x.page_number)
        
        # BATCHING: Split large books into chunks of 20 pages to prevent OOM
        # LightRAG handles relationships across calls, but memory is finite
        BATCH_PAGES = 20
        failed = False
        
        for j in range(0, len(grp), BATCH_PAGES):
            batch_chunks = grp[j:j+BATCH_PAGES]
            batch_text = "\n".join([c.text for c in batch_chunks])
            try:
                await rag.ainsert(batch_text)
            except (RuntimeError, ValueError, OSError) as e:
                logger.error(f"Failed to insert batch {j} of {pdf}: {e}")
                failed = True
                break
        
        if not failed:
            # Update manifest specifically for LightRAG, commit per-PDF (Commit-on-Success)
            update_manifest("lightrag", pdf, grp[0].file_hash)
            
        # MEMORY GC (Every 50 PDFs)
        if i > 0 and i % 50 == 0:
            gc.collect()

    return rag


# --- HELPER: MANUAL STALE CLEANUP ---
def purge_stale_data(collection, manifest_name="chroma"):
    """Manual utility to find and remove orphaned chunks not in manifest."""
    logger.info("Scanning for orphaned data...")
    manifest = load_manifest(manifest_name)
    valid_sources = set(manifest.keys())

    from rag_pipeline.chroma_pipeline import purge_stale_data as purge_stale_data_pkg

    deleted = purge_stale_data_pkg(collection, valid_sources=valid_sources, logger=logger)
    if deleted:
        logger.info("Cleanup complete.")


## 5. Phase 5: Async-Safe Pipeline & Evaluation
Refactored to allow direct `await` calls in Notebooks.

In [ ]:
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall
from langchain_groq import ChatGroq
from langchain_community.embeddings import HuggingFaceEmbeddings
from datasets import Dataset
from lightrag import QueryParam
from sentence_transformers import CrossEncoder

# Initialize shared resources lazily with safe env validation
_groq_evaluator_instance = None
_hf_embeddings_instance = None
_reranker_instance = None

def get_eval_components():
    global _groq_evaluator_instance, _hf_embeddings_instance, _reranker_instance

    api_key = os.environ.get("GROQ_API_KEY")
    if not api_key:
        raise ValueError("GROQ_API_KEY is required for evaluation. Export it before running evaluation.")

    if _groq_evaluator_instance is None:
        _groq_evaluator_instance = ChatGroq(model="llama-3.3-70b-versatile", api_key=api_key)

    if _hf_embeddings_instance is None:
        _hf_embeddings_instance = HuggingFaceEmbeddings(model_name="BAAI/bge-small-en-v1.5")

    if _reranker_instance is None:
        device_str = "cuda" if torch.cuda.is_available() else "cpu"
        print(f"Reranker using device: {device_str}")
        _reranker_instance = CrossEncoder("BAAI/bge-reranker-large", device=device_str)

    return _groq_evaluator_instance, _hf_embeddings_instance, _reranker_instance

# --- 30 REAL QA PAIRS FOR EVALUATION ---
from rag_pipeline.evaluation import generate_test_dataset


from rag_pipeline.query_engine import (
    generate_trading_answer_robust as generate_trading_answer_robust_pkg,
    run_evaluation_pipeline as run_evaluation_pipeline_pkg,
)


async def generate_trading_answer_robust(query: str, collection, rag_instance=None):
    """End-to-end RAG with BM25 hybrid retrieval."""
    return await generate_trading_answer_robust_pkg(
        query,
        collection,
        rag_instance,
        bm25_index=bm25_index,
        bm25_id_map=bm25_id_map,
        retrieval_top_k=RETRIEVAL_TOP_K,
        rrf_k=RRF_K,
        rerank_candidates=RERANK_CANDIDATES,
        rerank_batch_size=RERANK_BATCH_SIZE,
        final_top_k=FINAL_TOP_K,
        context_max_chars=CONTEXT_MAX_CHARS,
        tokenize_for_bm25_fn=tokenize_for_bm25,
        get_eval_components_fn=get_eval_components,
        get_cached_groq_client_fn=get_cached_groq_client,
        query_param_factory=lambda: QueryParam(mode="hybrid"),
        logger=logger,
    )


async def run_evaluation_pipeline(questions, ground_truths, collection, rag_instance):
    """Async wrapper for RAGAS evaluation."""
    print("Generating Answers for Eval...")
    results = await run_evaluation_pipeline_pkg(
        questions,
        ground_truths,
        collection,
        rag_instance,
        generate_answer_fn=generate_trading_answer_robust,
        dataset_from_dict_fn=Dataset.from_dict,
        evaluate_fn=evaluate,
        metrics=[faithfulness, answer_relevancy, context_precision, context_recall],
        get_eval_components_fn=get_eval_components,
        throttle_sec=EVAL_THROTTLE_SEC,
        progress_fn=lambda items: tqdm(items),
    )
    print("Evaluation complete.")
    return results

# USAGE EXAMPLE (Uncomment to run):
# if 'enriched_chunks' in locals():
#     coll = populate_chromadb(enriched_chunks)
#     rag = await populate_lightrag(enriched_chunks)
#     test_data = generate_test_dataset()
#     eval_res = await run_evaluation_pipeline(test_data["questions"], test_data["ground_truths"], coll, rag)
#     print(eval_res)